# Matched-pairs drift: single patch vs. full PICS (Libya 4)

Does the per-channel $\Delta R$-vs-$\Delta t$ relationship look **better** when a
single PICS is analysed with its full patch pool instead of one patch?

This notebook puts the two approaches side by side for Libya 4, per channel, for
both products (L1B radiance, L2A reflectance):

* **Single patch** (`10_7_1`): each point is one geometry-matched acquisition
  pair for that one patch (`n ≈ 9`). Read straight from the per-patch delta table.
* **Full PICS** (aggregated, option A): each point is one *unique acquisition
  pair*, with its spatial-replicate patches collapsed to a median $\Delta R$
  (`n ≈ 37`). Markers are sized by `n_patches` (replication) and carry whiskers =
  the across-patch min/max spread.

Both panels overlay the same **Theil–Sen** fit used in
`run_matched_pairs_drift_analysis.py`, with drift rate, 95% CI, Kendall $p$ and
$n$. A tighter CI / lower $p$ / less scatter on the right is evidence that
pooling the full PICS sharpens the drift estimate.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines


def find_repo_root(start=None):
    """Walk up from the notebook's cwd until the emit_pics/ package is found."""
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'emit_pics').is_dir():
            return cand
    raise RuntimeError('Could not locate repo root (no emit_pics/ upward from cwd).')


REPO = find_repo_root()
sys.path.insert(0, str(REPO))

# Reuse the pipeline's own fit + marker-size helpers so this parallels the
# per-channel drift figures exactly.
from emit_pics.matched_pairs_drift_analysis import (
    fit_channel_drift, DAYS_PER_YEAR, n_patches_marker_size)

# ---------------- USER-CONFIGURABLE INPUTS ----------------
PICS_ID         = 'Libya_4'
SINGLE_PATCH_ID = '10_7_1'      # the single patch to contrast against the pool
PATCH_SIZE      = '10'          # patch-size token (names the aggregated table)
TARGET_WLS      = [490, 660, 860, 1600]   # nm; nearest channel is used
SAVE_FIGS       = False         # also write PNGs to figures/single_vs_full_PICS/

SINGLE_PATH = REPO / 'data' / 'delta_r' / 'per_patch' / f'{PICS_ID}_{SINGLE_PATCH_ID}_delta.csv'
AGG_PATH    = REPO / 'data' / 'delta_r' / 'aggregated' / f'{PICS_ID}_{PATCH_SIZE}_delta.csv'
FIG_DIR     = REPO / 'figures' / 'single_vs_full_PICS'
print('repo root:', REPO)

In [ ]:
# Load both delta tables (fail loudly if a stage hasn't been run yet).
for pth, how in ((SINGLE_PATH, 'run_delta_calculations.py (or the batched runner)'),
                 (AGG_PATH, 'run_PICS_allpatches_aggregate_pairs.py')):
    if not pth.exists():
        raise FileNotFoundError(f'Missing {pth}\n  -> produce it with {how}.')

single_df = pd.read_csv(SINGLE_PATH)
agg_df    = pd.read_csv(AGG_PATH)

print(f'single patch {SINGLE_PATCH_ID}: {len(single_df)} rows | '
      f"{single_df['pair_id'].nunique()} pair(s) | "
      f"{single_df['band_index'].nunique()} channel(s)")
print(f'full PICS aggregated : {len(agg_df)} rows | '
      f"{agg_df['pair_id'].nunique()} pair(s) | "
      f"{agg_df['band_index'].nunique()} channel(s) | "
      f"n_patches median {agg_df['n_patches'].median():.0f}, max {agg_df['n_patches'].max()}")

## Resolve target wavelengths to channels

Channels are keyed by `band_index`; each target wavelength is mapped to its
nearest band from the aggregated table's `band_index`→`wavelength` lookup.

In [ ]:
def nearest_band(target_nm, table):
    """band_index whose wavelength is closest to target_nm."""
    m = table.drop_duplicates('band_index')[['band_index', 'wavelength']]
    i = (m['wavelength'] - target_nm).abs().idxmin()
    return int(m.loc[i, 'band_index'])


bands = [nearest_band(w, agg_df) for w in TARGET_WLS]
wl_of = agg_df.drop_duplicates('band_index').set_index('band_index')['wavelength']
for tgt, b in zip(TARGET_WLS, bands):
    abs_flag = bool(agg_df.loc[agg_df['band_index'] == b, 'absorption_band_flag'].astype(str)
                    .str.upper().isin(['TRUE', '1']).any())
    note = '  (WATER-VAPOR absorption band!)' if abs_flag else ''
    print(f'target {tgt:>5} nm  ->  band {b:>3}  ({wl_of[b]:.1f} nm){note}')

## Plot helpers

`compare_channel` builds the 2×2 figure for one channel: rows = RAD / RFL,
columns = single patch / full PICS. `draw_fit` overlays the Theil–Sen line and
the stats box, reusing `fit_channel_drift` from the pipeline.

In [ ]:
PROD = {'rad': ('Radiance (L1B)', '#1f77b4'),
        'rfl': ('Reflectance (L2A)', '#2ca02c')}


def draw_fit(ax, stats, x, color='k'):
    """Overlay the Theil–Sen line + a stats box (drift, 95% CI, Kendall p, n)."""
    n = int(stats.get('n', 0) or 0)
    slope = stats.get('slope_pct_yr', np.nan)
    x = np.asarray(x, dtype=float)
    if x.size and np.isfinite(slope):
        xr = np.array([np.nanmin(x), np.nanmax(x)])
        yline = stats['intercept_pct'] + (slope / DAYS_PER_YEAR) * xr
        ax.plot(xr, yline, color=color, lw=1.6, zorder=5)
        txt = (f"drift = {slope:+.3f} %/yr\n"
               f"95% CI [{stats['ci_low_pct_yr']:+.2f}, {stats['ci_high_pct_yr']:+.2f}]\n"
               f"Kendall p = {stats['p_value']:.2g}   n = {n}")
    else:
        txt = f'no fit (n={n})'
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, va='top', ha='left',
            fontsize=8, family='monospace',
            bbox=dict(boxstyle='round', fc='white', ec='0.7', alpha=0.85))


def compare_channel(bidx):
    """2×2 single-patch vs full-PICS comparison for one channel; returns (fig, wl)."""
    ss = single_df[single_df['band_index'] == bidx]
    ff = agg_df[agg_df['band_index'] == bidx]
    wl = float(np.nanmedian(pd.concat([ss['wavelength'], ff['wavelength']])))
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey='row')
    for i, (prod, (label, color)) in enumerate(PROD.items()):
        dcol = f'delta_{prod}_pct_median'
        # --- left: single patch (points, fixed marker size) ---
        axL = axes[i, 0]
        axL.axhline(0, color='gray', lw=0.8, ls='--')
        axL.scatter(ss['dt_days'], ss[dcol], s=45, c=color,
                    edgecolors='k', linewidths=0.4, zorder=3)
        draw_fit(axL, fit_channel_drift(ss['dt_days'], ss[dcol]),
                 ss['dt_days'].to_numpy())
        # --- right: full PICS (median + min/max whiskers, marker ∝ n_patches) ---
        axR = axes[i, 1]
        axR.axhline(0, color='gray', lw=0.8, ls='--')
        med, lo, hi = ff[dcol], ff[f'{dcol}_min'], ff[f'{dcol}_max']
        yerr = np.clip(np.vstack([med - lo, hi - med]), 0, None)
        axR.errorbar(ff['dt_days'], med, yerr=yerr, fmt='none',
                     ecolor='0.6', elinewidth=0.7, capsize=2, zorder=1)
        axR.scatter(ff['dt_days'], med, s=n_patches_marker_size(ff['n_patches']),
                    c=color, edgecolors='k', linewidths=0.4, zorder=3)
        draw_fit(axR, fit_channel_drift(ff['dt_days'], ff[dcol]),
                 ff['dt_days'].to_numpy())
        axes[i, 0].set_ylabel(f'ΔR {prod} — % change (median)')
    # marker-size reference on the top-right panel
    handles = [mlines.Line2D([], [], color='gray', marker='o', ls='',
                             markeredgecolor='k',
                             markersize=np.sqrt(n_patches_marker_size(n)),
                             label=f'{n} patch(es)') for n in (1, 5, 10)]
    axes[0, 1].legend(handles=handles, fontsize=7, loc='lower left',
                      framealpha=0.9, title='marker size')
    axes[0, 0].set_title(f'single patch {SINGLE_PATCH_ID}  (n={len(ss)})')
    axes[0, 1].set_title(f'full {PICS_ID} PICS  (n={len(ff)})')
    axes[1, 0].set_xlabel('Δt — days between pair')
    axes[1, 1].set_xlabel('Δt — days between pair')
    fig.suptitle(f'Channel {bidx} — {wl:.0f} nm    '
                 f'(right: marker ∝ #patches, whiskers = across-patch min/max; '
                 f'black line = Theil–Sen fit)', fontsize=11)
    fig.tight_layout()
    return fig, wl

## Render one comparison figure per curated channel

In [ ]:
if SAVE_FIGS:
    FIG_DIR.mkdir(parents=True, exist_ok=True)

for b in bands:
    fig, wl = compare_channel(b)
    if SAVE_FIGS:
        out = FIG_DIR / f'{PICS_ID}_{int(round(wl)):04d}nm_single_vs_full.png'
        fig.savefig(out, dpi=200, bbox_inches='tight')
        print('saved', out)
    plt.show()

## Quantify “better”

Side-by-side Theil–Sen fit stats per curated channel and product. **`ci_width`**
(the 95% CI span on the drift rate) and the Kendall **`p`** are the headline
comparison: if the full-PICS pool sharpens the estimate, its `ci_width` shrinks
and its `p` drops relative to the single patch.

In [ ]:
rows = []
for b in bands:
    wl = float(wl_of[b])
    for prod in ('rad', 'rfl'):
        dcol = f'delta_{prod}_pct_median'
        s = fit_channel_drift(single_df.loc[single_df['band_index'] == b, 'dt_days'],
                              single_df.loc[single_df['band_index'] == b, dcol])
        f = fit_channel_drift(agg_df.loc[agg_df['band_index'] == b, 'dt_days'],
                              agg_df.loc[agg_df['band_index'] == b, dcol])
        rows.append({
            'wl_nm': round(wl), 'product': prod,
            'single_n': s['n'], 'single_slope': round(s['slope_pct_yr'], 3),
            'single_ci_width': round(s['ci_high_pct_yr'] - s['ci_low_pct_yr'], 3),
            'single_p': round(s['p_value'], 4),
            'full_n': f['n'], 'full_slope': round(f['slope_pct_yr'], 3),
            'full_ci_width': round(f['ci_high_pct_yr'] - f['ci_low_pct_yr'], 3),
            'full_p': round(f['p_value'], 4),
        })
compare = pd.DataFrame(rows)
compare['ci_shrunk'] = compare['full_ci_width'] < compare['single_ci_width']
compare

### How to read this

* **Left vs. right, same row.** Left is one patch's 9 matched pairs; right is the
  whole PICS collapsed to ~37 unique acquisition pairs. Same physical channel,
  same $\Delta R$-vs-$\Delta t$ model.
* **Whiskers (right only)** are the across-patch min/max for each acquisition
  pair — how much the spatial replicates disagreed before the median collapse.
  **Marker size (right only)** is `n_patches`: bigger dots rest on more replicates.
* **The single patch is not a subset artifact.** Its 9 pairs are among the 37, so
  the right panel is the same site with more of its own observations, not a
  different target.
* **“Better” = a sharper, more trustworthy slope**, i.e. a narrower 95% CI
  (`ci_shrunk == True`) and a lower Kendall $p$, without the slope itself
  lurching — which would say the single patch was just noisy, not biased. A slope
  that *moves* a lot between the columns is a flag that one patch was
  unrepresentative.

Note this is still the geometry-matched estimator on a per-PICS pool; the
cross-PICS drift step and the BRDF (full-pool) approach are separate.